In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")


In [ ]:
import kagglehub

path = kagglehub.dataset_download("carrie1/ecommerce-data")
print(path)


In [ ]:
df = pd.read_csv(path + "/data.csv", encoding="ISO-8859-1")
df.shape


In [ ]:
df.head()


In [ ]:
df = df.dropna(subset=['CustomerID'])
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']
df.shape


In [ ]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
})
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()


In [ ]:
rfm.describe()


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, rfm.columns):
    sns.histplot(rfm[col], bins=40, ax=ax)
    ax.set_title(col)
plt.show()


In [ ]:
rfm_log = np.log1p(rfm)

scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm_log)
rfm_scaled = pd.DataFrame(rfm_scaled, index=rfm.index, columns=rfm.columns)
rfm_scaled.head()


In [ ]:
inertias = []
sil_scores = []
K_range = range(2, 10)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(rfm_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(rfm_scaled, labels, sample_size=3000, random_state=42))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, marker='o')
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[1].plot(K_range, sil_scores, marker='o', color='green')
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("k")
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['KMeans_Cluster'] = kmeans.fit_predict(rfm_scaled)

rfm['KMeans_Cluster'].value_counts().sort_index()


In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=15)
rfm['DBSCAN_Cluster'] = dbscan.fit_predict(rfm_scaled)

print("Clusters found:", len(set(rfm['DBSCAN_Cluster'])) - (1 if -1 in rfm['DBSCAN_Cluster'].values else 0))
print("Noise points:", (rfm['DBSCAN_Cluster'] == -1).sum())
rfm['DBSCAN_Cluster'].value_counts().sort_index()


In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(rfm_scaled)

rfm['PCA1'] = pca_result[:, 0]
rfm['PCA2'] = pca_result[:, 1]

print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained:", pca.explained_variance_ratio_.sum())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(x='PCA1', y='PCA2', hue='KMeans_Cluster', data=rfm, palette='tab10', s=15, ax=axes[0])
axes[0].set_title("K-Means Clusters (PCA)")

sns.scatterplot(x='PCA1', y='PCA2', hue='DBSCAN_Cluster', data=rfm, palette='tab10', s=15, ax=axes[1])
axes[1].set_title("DBSCAN Clusters (PCA)")

plt.show()


In [ ]:
sample_idx = rfm.sample(n=min(3000, len(rfm)), random_state=42).index
sample_scaled = rfm_scaled.loc[sample_idx]

tsne = TSNE(n_components=2, perplexity=30, random_state=42)
tsne_result = tsne.fit_transform(sample_scaled)

tsne_df = pd.DataFrame(tsne_result, index=sample_idx, columns=['TSNE1', 'TSNE2'])
tsne_df['KMeans_Cluster'] = rfm.loc[sample_idx, 'KMeans_Cluster']
tsne_df['DBSCAN_Cluster'] = rfm.loc[sample_idx, 'DBSCAN_Cluster']
tsne_df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.scatterplot(x='TSNE1', y='TSNE2', hue='KMeans_Cluster', data=tsne_df, palette='tab10', s=15, ax=axes[0])
axes[0].set_title("K-Means Clusters (t-SNE)")

sns.scatterplot(x='TSNE1', y='TSNE2', hue='DBSCAN_Cluster', data=tsne_df, palette='tab10', s=15, ax=axes[1])
axes[1].set_title("DBSCAN Clusters (t-SNE)")

plt.show()


In [ ]:
profile = rfm.groupby('KMeans_Cluster').agg({
    'Recency': 'mean',
    'Frequency': 'mean',
    'Monetary': 'mean',
    'KMeans_Cluster': 'count'
}).rename(columns={'KMeans_Cluster': 'Customers'}).round(2)

profile


In [ ]:
profile_melt = profile[['Recency', 'Frequency', 'Monetary']].reset_index().melt(id_vars='KMeans_Cluster')

g = sns.FacetGrid(profile_melt, col='variable', sharey=False, height=4)
g.map_dataframe(sns.barplot, x='KMeans_Cluster', y='value', palette='tab10')
plt.show()


In [ ]:
avg_r = rfm['Recency'].mean()
avg_f = rfm['Frequency'].mean()
avg_m = rfm['Monetary'].mean()

def assign_persona(row):
    recent = row['Recency'] < avg_r
    frequent = row['Frequency'] > avg_f
    big_spender = row['Monetary'] > avg_m
    if recent and frequent and big_spender:
        return "Champions"
    elif recent and not frequent:
        return "New / Promising Customers"
    elif not recent and (frequent or big_spender):
        return "At-Risk Big Spenders"
    elif recent:
        return "Loyal Regulars"
    else:
        return "Lost / Hibernating Customers"

profile['Persona'] = profile.apply(assign_persona, axis=1)
profile


In [ ]:
marketing_actions = {
    "Champions": "Reward with VIP perks, early access to new products, and referral programs.",
    "Loyal Regulars": "Upsell and cross-sell, offer loyalty points and bundle deals.",
    "New / Promising Customers": "Send welcome series, onboarding offers, and encourage a second purchase.",
    "At-Risk Big Spenders": "Run win-back campaigns with personalized discounts and 'we miss you' emails.",
    "Lost / Hibernating Customers": "Low-cost reactivation emails or clearance offers; stop heavy spending on them."
}

profile['Marketing_Action'] = profile['Persona'].map(marketing_actions)

for cluster_id, row in profile.iterrows():
    print(f"Cluster {cluster_id} -> {row['Persona']}")
    print(f"  Customers: {int(row['Customers'])} | Recency: {row['Recency']} days | Frequency: {row['Frequency']} orders | Monetary: {row['Monetary']}")
    print(f"  Action: {row['Marketing_Action']}\n")


In [ ]:
rfm['Persona'] = rfm['KMeans_Cluster'].map(profile['Persona'])

plt.figure(figsize=(8, 6))
sns.scatterplot(x='PCA1', y='PCA2', hue='Persona', data=rfm, palette='tab10', s=15)
plt.title("Customer Personas (PCA View)")
plt.show()
